# Multi-Agent Research with Google's A2A Protocol

Three agents, built with **three different frameworks**, collaborate on a research question. They never import each other's code. Each one runs as its own **A2A server**, publishes an **Agent Card** describing what it can do, and is called over HTTP with standard **A2A messages**. A **LangGraph** orchestrator discovers them by skill and coordinates the workflow.

| Agent | Built with | A2A skill | Job |
|---|---|---|---|
| Task Extractor | **LangChain** (structured output) | `extract_research_brief` | Turns the user's request into a research brief with search queries, or asks a clarifying question |
| Researcher | **CrewAI** + Tavily web search | `web_research` | Searches the web and returns findings, each tied to a source URL |
| Writer | **Plain OpenAI / Anthropic SDK** (no framework) | `write_cited_answer` | Writes the final answer, citing only the sources it was given |
| Orchestrator | **LangGraph** | (A2A client) | Discovers the agents, sends each one a message, routes on the result |

**Why A2A matters:** the orchestrator only knows the protocol. Any agent, in any framework or language, can be swapped in as long as it speaks A2A. Step 6 proves this by running the same orchestrator against stub agents before the real ones exist.

**Setup:** add `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`) and `TAVILY_API_KEY` under Colab **Secrets** (🔑) with notebook access on. Steps 1–6 run without any keys.

## Step 1: Install packages


In [1]:
!pip install -q -U langgraph langchain-core langchain-openai langchain-anthropic "crewai[anthropic]" openai anthropic tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: A2A in five ideas

A2A (Agent2Agent) is an open protocol, started by Google and now hosted by the Linux Foundation, for agents to talk to **other agents** across frameworks, vendors and servers. It is built on ordinary web standards: HTTP, JSON-RPC 2.0 and (for streaming) server-sent events.

| Concept | What it is | In this notebook |
|---|---|---|
| **Agent Card** | A JSON "business card" each agent publishes at `/.well-known/agent-card.json`: name, URL, capabilities and **skills** | The orchestrator reads the cards and picks agents by skill ID, not by hard-coded address |
| **Message** | One turn of communication, with a `role` (`user` or `agent`) and a list of **parts** | The orchestrator sends messages to each agent |
| **Part** | A piece of content: `text`, `data` (structured JSON) or `file` | Agents pass both a readable `text` part and a machine-readable `data` part |
| **Task** | The unit of work an agent creates for a request, with a **lifecycle state**: `submitted → working → completed`, or `input-required`, `failed`, `rejected`, `canceled` | The orchestrator routes on the state, e.g. stops and asks the user when a task is `input-required` |
| **Artifact** | The output a completed task produces, made of parts | The brief, the findings and the final answer |

A request is a JSON-RPC call to the agent's URL:

```json
{"jsonrpc": "2.0", "id": "1", "method": "message/send",
 "params": {"message": {"kind": "message", "messageId": "m-1", "role": "user",
                        "parts": [{"kind": "text", "text": "Can you summarize the latest trends in GenAI for enterprise?"}]}}}
```

and the reply is a Task:

```json
{"jsonrpc": "2.0", "id": "1",
 "result": {"kind": "task", "id": "t-9", "contextId": "c-3",
            "status": {"state": "completed"},
            "artifacts": [{"artifactId": "a-1", "parts": [{"kind": "text", "text": "..."}]}]}}
```

**A2A vs. MCP:** MCP connects an agent to its **tools and data** (a database, an API). A2A connects an agent to **other agents**, which have their own reasoning and may take many steps. They are complementary: an A2A agent often uses MCP tools internally.

**About this implementation:** Steps 3 and 4 implement the A2A wire format directly with Python's standard library (about 150 lines), so every message is visible. The official [`a2a-sdk`](https://github.com/a2aproject/a2a-python) wraps the same messages and adds streaming, push notifications and authentication.

## Step 3: The A2A server side

`A2AServer` wraps **any** Python function as an A2A agent. It:

- serves the Agent Card at `GET /.well-known/agent-card.json`;
- accepts JSON-RPC `message/send` at `POST /`, creates a **Task**, runs the agent, and returns the task with its final state and artifacts;
- supports `tasks/get`, so a client can look a task up again later;
- turns errors into standard JSON-RPC error codes, and an agent crash into a `failed` task rather than a broken connection.

Each server runs in a background thread on a free local port.

In [5]:
import json
import threading
import uuid
import urllib.error
import urllib.request
from dataclasses import dataclass
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

A2A_VERSION = "0.3.0"
CARD_PATH = "/.well-known/agent-card.json"   # the well-known discovery path in A2A 0.3

# Standard JSON-RPC 2.0 error codes, plus A2A's own
PARSE_ERROR, INVALID_REQUEST, METHOD_NOT_FOUND, INVALID_PARAMS = -32700, -32600, -32601, -32602
TASK_NOT_FOUND = -32001


def new_id() -> str:
    return str(uuid.uuid4())


def now() -> str:
    return datetime.now(timezone.utc).isoformat()


# ---------- A2A building blocks: parts, messages, skills, Agent Cards ----------
def text_part(text: str) -> dict:
    return {"kind": "text", "text": text}


def data_part(data: dict) -> dict:
    return {"kind": "data", "data": data}       # structured JSON another agent can use directly


def make_message(role: str, parts: list, context_id: str | None = None, task_id: str | None = None) -> dict:
    message = {"kind": "message", "messageId": new_id(), "role": role, "parts": parts}
    if context_id:
        message["contextId"] = context_id       # groups related tasks into one conversation
    if task_id:
        message["taskId"] = task_id
    return message


def make_skill(skill_id: str, name: str, description: str, tags: list, examples: list = ()) -> dict:
    return {"id": skill_id, "name": name, "description": description, "tags": list(tags), "examples": list(examples)}


def make_agent_card(name: str, description: str, skills: list) -> dict:
    """The agent's public description. `url` is filled in when the server starts."""
    return {
        "protocolVersion": A2A_VERSION,
        "name": name,
        "description": description,
        "url": "",
        "version": "1.0.0",
        "preferredTransport": "JSONRPC",
        "capabilities": {"streaming": False, "pushNotifications": False},
        "defaultInputModes": ["text/plain", "application/json"],
        "defaultOutputModes": ["text/plain", "application/json"],
        "skills": skills,
    }


@dataclass
class AgentReply:
    """What an agent function returns. The server turns it into an A2A Task."""
    state: str = "completed"          # completed | input-required | failed | rejected
    text: str = ""
    data: dict | None = None


def rpc_result(request_id, result: dict) -> dict:
    return {"jsonrpc": "2.0", "id": request_id, "result": result}


def rpc_error(request_id, code: int, message: str) -> dict:
    return {"jsonrpc": "2.0", "id": request_id, "error": {"code": code, "message": message}}


# ---------- The server ----------
class A2AServer:
    """Serves one agent over A2A. `agent_fn(text, data) -> AgentReply` is the agent itself."""

    def __init__(self, card: dict, agent_fn, host: str = "127.0.0.1", port: int = 0):
        self.card, self.agent_fn, self.tasks = card, agent_fn, {}
        server = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):          # keep the notebook output clean
                pass

            def _send(self, status: int, body: dict):
                raw = json.dumps(body).encode()
                self.send_response(status)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(raw)))
                self.end_headers()
                self.wfile.write(raw)

            def do_GET(self):                       # discovery: anyone can read the Agent Card
                if self.path == CARD_PATH:
                    self._send(200, server.card)
                else:
                    self._send(404, {"error": "not found"})

            def do_POST(self):                      # all A2A calls are JSON-RPC POSTs to the agent URL
                body = self.rfile.read(int(self.headers.get("Content-Length", 0)))
                self._send(200, server.handle_rpc(body))

        self.httpd = ThreadingHTTPServer((host, port), Handler)     # port 0 = pick any free port
        self.url = f"http://{host}:{self.httpd.server_address[1]}/"
        self.card["url"] = self.url
        self._thread = threading.Thread(target=self.httpd.serve_forever, daemon=True)

    def start(self):
        self._thread.start()
        return self

    def stop(self):
        self.httpd.shutdown()
        self.httpd.server_close()

    # ---- JSON-RPC dispatch ----
    def handle_rpc(self, raw: bytes) -> dict:
        try:
            request = json.loads(raw)
        except json.JSONDecodeError:
            return rpc_error(None, PARSE_ERROR, "Parse error: the body is not valid JSON")
        if not isinstance(request, dict) or request.get("jsonrpc") != "2.0" or "method" not in request:
            return rpc_error(request.get("id") if isinstance(request, dict) else None, INVALID_REQUEST,
                             "Invalid request: expected a JSON-RPC 2.0 object with a method")
        request_id, method, params = request.get("id"), request["method"], request.get("params") or {}
        if method == "message/send":
            return self._message_send(request_id, params)
        if method == "tasks/get":
            task = self.tasks.get(params.get("id"))
            return rpc_result(request_id, task) if task else rpc_error(request_id, TASK_NOT_FOUND, "Task not found")
        return rpc_error(request_id, METHOD_NOT_FOUND, f"Method not found: {method}")

    def _message_send(self, request_id, params: dict) -> dict:
        message = params.get("message")
        if not isinstance(message, dict) or not message.get("parts"):
            return rpc_error(request_id, INVALID_PARAMS, "params.message with at least one part is required")
        text = "\n".join(p.get("text", "") for p in message["parts"] if p.get("kind") == "text")
        data = next((p.get("data") for p in message["parts"] if p.get("kind") == "data"), None)

        # 1. Create the Task. Its lifecycle starts in "working".
        context_id = message.get("contextId") or new_id()
        task = {"kind": "task", "id": new_id(), "contextId": context_id,
                "status": {"state": "working", "timestamp": now()}, "artifacts": [], "history": [message]}
        self.tasks[task["id"]] = task

        # 2. Run the agent. A crash becomes a "failed" task the client can handle.
        try:
            reply = self.agent_fn(text, data)
        except Exception as err:
            reply = AgentReply(state="failed", text=f"{type(err).__name__}: {err}")

        # 3. Record the outcome. A completed task carries its output as an Artifact;
        #    any other state carries an agent Message explaining it (e.g. the clarifying question).
        parts = [text_part(reply.text)] + ([data_part(reply.data)] if reply.data is not None else [])
        if reply.state == "completed":
            task["artifacts"].append({"artifactId": new_id(), "name": "result", "parts": parts})
            task["status"] = {"state": "completed", "timestamp": now()}
        else:
            task["status"] = {"state": reply.state, "timestamp": now(),
                              "message": make_message("agent", parts, context_id, task["id"])}
        return rpc_result(request_id, task)

## Step 4: The A2A client side

The orchestrator needs only three things: read an Agent Card, send a message, and read the resulting Task. `AgentDirectory` adds **discovery**: register agents by URL, then look them up **by skill**, so the orchestrator never hard-codes which agent lives where.

Every request and response is saved in `WIRE_LOG`, so Step 11 can show the exact JSON that crossed the wire.

In [2]:
WIRE_LOG = []


class A2AError(Exception):
    """A JSON-RPC error returned by an agent, or a failure to reach it."""


def _http_json(url: str, body: dict | None = None, timeout: float = 300) -> dict:
    request = urllib.request.Request(
        url, data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"}, method="GET" if body is None else "POST")
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read())
    except (urllib.error.URLError, TimeoutError) as err:
        raise A2AError(f"could not reach {url}: {err}") from err


def fetch_agent_card(base_url: str) -> dict:
    return _http_json(base_url.rstrip("/") + CARD_PATH)


def send_message(agent_url: str, text: str, data: dict | None = None, context_id: str | None = None,
                 timeout: float = 300) -> dict:
    """Send an A2A message/send request and return the Task."""
    parts = [text_part(text)] + ([data_part(data)] if data is not None else [])
    request = {"jsonrpc": "2.0", "id": new_id(), "method": "message/send",
               "params": {"message": make_message("user", parts, context_id)}}
    response = _http_json(agent_url, request, timeout)
    WIRE_LOG.append({"agent_url": agent_url, "request": request, "response": response})
    if "error" in response:
        raise A2AError(f"JSON-RPC error {response['error']['code']}: {response['error']['message']}")
    return response["result"]


# ---------- Reading a Task ----------
def task_state(task: dict) -> str:
    return task["status"]["state"]


def task_parts(task: dict) -> list:
    """Output parts: the artifacts of a completed task, or the status message of any other state."""
    if task.get("artifacts"):
        return [part for artifact in task["artifacts"] for part in artifact["parts"]]
    return (task["status"].get("message") or {}).get("parts", [])


def task_text(task: dict) -> str:
    return "\n".join(p["text"] for p in task_parts(task) if p["kind"] == "text")


def task_data(task: dict) -> dict | None:
    return next((p["data"] for p in task_parts(task) if p["kind"] == "data"), None)


# ---------- Discovery ----------
class AgentDirectory:
    """Knows agents only through their Agent Cards, and finds them by skill ID."""

    def __init__(self):
        self.cards = []

    def register(self, base_url: str) -> dict:
        card = fetch_agent_card(base_url)
        self.cards = [c for c in self.cards if c["url"] != card["url"]] + [card]
        return card

    def find(self, skill_id: str) -> dict:
        for card in self.cards:
            if any(s["id"] == skill_id for s in card["skills"]):
                return card
        raise A2AError(f"no registered agent offers the skill '{skill_id}'")

    def show(self):
        for card in self.cards:
            skills = ", ".join(s["id"] for s in card["skills"])
            print(f"  {card['name']:<16} {card['url']:<26} skills: {skills}")

## Step 5: The orchestrator (LangGraph, acting as an A2A client)

The graph has one node per step. Each node finds the right agent **by skill** and sends it an A2A message. After each step, a conditional edge reads the **task state**:

```
START → extract ──completed──→ research ──completed──→ write → END
           │                       │
           ├─ input-required ──→ END (ask the user the agent's question)
           └─ failed ──────────→ END (report which agent failed and why)
```

All three messages share one `contextId`, which tells the agents they belong to the same piece of work. The `trace` records every A2A call: which agent, which skill, the final task state and how long it took.

In [3]:
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class ResearchState(TypedDict, total=False):
    question: str
    context_id: str
    brief: dict
    findings: list
    answer: str
    sources: list
    status: str                                   # set when the run stops: completed | input-required | failed
    message: str                                  # the clarifying question or error, if stopped early
    trace: Annotated[list, operator.add]          # each node appends its A2A calls


def build_orchestrator(directory: AgentDirectory):
    def call(skill_id: str, text: str, data: dict | None, state: ResearchState):
        card = directory.find(skill_id)                          # discovery by skill, not by address
        start = time.perf_counter()
        try:
            task = send_message(card["url"], text, data, context_id=state["context_id"])
        except A2AError as err:
            task = {"status": {"state": "failed", "message": make_message("agent", [text_part(str(err))])},
                    "artifacts": []}
        entry = {"agent": card["name"], "skill": skill_id, "state": task_state(task),
                 "seconds": round(time.perf_counter() - start, 1)}
        return task, entry

    def stopped(task, entry) -> dict | None:
        """If the task did not complete, stop the run and pass the agent's message on."""
        if task_state(task) == "completed":
            return None
        return {"status": task_state(task), "message": f"{entry['agent']}: {task_text(task)}", "trace": [entry]}

    def extract(state: ResearchState) -> dict:
        task, entry = call("extract_research_brief", state["question"], None, state)
        return stopped(task, entry) or {"brief": task_data(task), "trace": [entry]}

    def research(state: ResearchState) -> dict:
        brief = state["brief"]
        task, entry = call("web_research", brief["task"], brief, state)
        return stopped(task, entry) or {"findings": task_data(task)["findings"], "trace": [entry]}

    def write(state: ResearchState) -> dict:
        payload = {"task": state["brief"]["task"], "audience": state["brief"].get("audience", "general"),
                   "findings": state["findings"]}
        task, entry = call("write_cited_answer", state["brief"]["task"], payload, state)
        result = task_data(task) or {}
        return stopped(task, entry) or {"answer": result.get("answer", task_text(task)),
                                        "sources": result.get("sources", []),
                                        "status": "completed", "trace": [entry]}

    def route(next_node: str):
        return lambda state: END if state.get("status") else next_node

    graph = StateGraph(ResearchState)
    graph.add_node("extract", extract)
    graph.add_node("research", research)
    graph.add_node("write", write)
    graph.add_edge(START, "extract")
    graph.add_conditional_edges("extract", route("research"), ["research", END])
    graph.add_conditional_edges("research", route("write"), ["write", END])
    graph.add_edge("write", END)
    return graph.compile()


def run(orchestrator, question: str, show: bool = True) -> dict:
    result = orchestrator.invoke({"question": question, "context_id": new_id(), "trace": []})
    if show:
        print("A2A calls:")
        for t in result["trace"]:
            print(f"  {t['agent']:<16} {t['skill']:<24} → {t['state']:<15} {t['seconds']:>6.1f} s")
        print(f"\nStatus: {result.get('status')}")
        if result.get("status") == "completed":
            print("\n" + result["answer"])
            print("\nSources:")
            for i, url in enumerate(result["sources"], 1):
                print(f"  [{i}] {url}")
        else:
            print(result.get("message"))
    return result

## Step 6: Offline tests with stub agents (no API keys)

Three **stub agents** with the same Agent Cards and skills as the real ones are started as A2A servers. The orchestrator from Step 5 runs against them unchanged. Later, the real agents replace the stubs and the orchestrator code doesn't change at all. That's the interoperability A2A promises.

The tests check the protocol (discovery, messages, JSON-RPC errors, task states, `tasks/get`) and the orchestrator's routing (completed, `input-required` and `failed` paths).

In [6]:
# Stop any servers left over from an earlier run of this notebook
for s in globals().get("SERVERS", []):
    s.stop()
SERVERS = []

# The three skills. The real agents in Step 8 publish exactly the same ones.
EXTRACT_SKILL = make_skill("extract_research_brief", "Extract research brief",
    "Turns a user request into a research task with web search queries, or asks a clarifying question.",
    ["planning", "intent"], ["Summarize the latest GenAI trends for enterprise"])
RESEARCH_SKILL = make_skill("web_research", "Web research",
    "Searches the web and returns findings, each tied to the URL it came from.",
    ["research", "web-search"], ["Recent enterprise GenAI adoption trends"])
WRITE_SKILL = make_skill("write_cited_answer", "Write cited answer",
    "Writes a concise answer for a given audience, citing only the findings it is given.",
    ["writing", "synthesis"])


# ---------- Stub agents: fixed behaviour, no LLM ----------
def stub_extractor(text, data):
    if len(text.split()) < 3:
        return AgentReply(state="input-required", text="What topic should I research?")
    return AgentReply(text="Research: " + text, data={"task": text, "search_queries": [text], "audience": "general",
                                                    "needs_clarification": False, "clarifying_question": ""})


def stub_researcher(text, data):
    if "crash" in text:
        raise RuntimeError("search service unavailable")
    return AgentReply(text="2 findings", data={"findings": [
        {"point": "Finding one", "source_url": "https://example.com/a"},
        {"point": "Finding two", "source_url": "https://example.com/b"}]})


def stub_writer(text, data):
    urls = [f["source_url"] for f in data["findings"]]
    return AgentReply(text="Answer [1][2]", data={"answer": "Finding one [1]. Finding two [2].", "sources": urls})


stubs = [A2AServer(make_agent_card("Stub Extractor", "Test double", [EXTRACT_SKILL]), stub_extractor).start(),
         A2AServer(make_agent_card("Stub Researcher", "Test double", [RESEARCH_SKILL]), stub_researcher).start(),
         A2AServer(make_agent_card("Stub Writer", "Test double", [WRITE_SKILL]), stub_writer).start()]
SERVERS += stubs

# 1. Discovery: cards are served at the well-known path and found by skill
stub_directory = AgentDirectory()
for s in stubs:
    stub_directory.register(s.url)
assert stub_directory.find("web_research")["name"] == "Stub Researcher"
assert fetch_agent_card(stubs[0].url)["protocolVersion"] == A2A_VERSION
print("✓ 1. discovery: Agent Cards served at", CARD_PATH, "and looked up by skill")

# 2. message/send: a completed Task with an artifact holding text and data parts
task = send_message(stubs[0].url, "GenAI trends for enterprise")
assert task["kind"] == "task" and task_state(task) == "completed"
assert task_data(task)["task"] == "GenAI trends for enterprise"
assert stubs[0].tasks[task["id"]]["status"]["state"] == "completed"
print("✓ 2. message/send: returns a completed Task whose artifact has text and data parts")

# 3. Task states: input-required carries the agent's question; a crash becomes a failed task
task = send_message(stubs[0].url, "trends")
assert task_state(task) == "input-required" and task_text(task) == "What topic should I research?"
task = send_message(stubs[1].url, "crash please")
assert task_state(task) == "failed" and "search service unavailable" in task_text(task)
print("✓ 3. task states: input-required and failed are reported, not raised")

# 4. JSON-RPC errors and tasks/get
assert stubs[0].handle_rpc(b"not json")["error"]["code"] == PARSE_ERROR
assert stubs[0].handle_rpc(json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tasks/cancelAll"}).encode())["error"]["code"] == METHOD_NOT_FOUND
assert stubs[0].handle_rpc(json.dumps({"jsonrpc": "2.0", "id": 1, "method": "message/send", "params": {}}).encode())["error"]["code"] == INVALID_PARAMS
task_id = next(iter(stubs[0].tasks))
looked_up = stubs[0].handle_rpc(json.dumps({"jsonrpc": "2.0", "id": 2, "method": "tasks/get", "params": {"id": task_id}}).encode())
assert looked_up["result"]["id"] == task_id
assert stubs[0].handle_rpc(json.dumps({"jsonrpc": "2.0", "id": 3, "method": "tasks/get", "params": {"id": "nope"}}).encode())["error"]["code"] == TASK_NOT_FOUND
try:
    stub_directory.find("translate")
    raise AssertionError("expected an error for an unknown skill")
except A2AError:
    pass
print("✓ 4. errors: parse, unknown method, bad params, unknown task and unknown skill all handled")

# 5. The orchestrator end to end, on all three paths
orchestrator = build_orchestrator(stub_directory)
done = run(orchestrator, "Summarize GenAI trends for enterprise", show=False)
assert done["status"] == "completed" and done["answer"] == "Finding one [1]. Finding two [2]."
assert [t["skill"] for t in done["trace"]] == ["extract_research_brief", "web_research", "write_cited_answer"]
assert len({e["request"]["params"]["message"]["contextId"] for e in WIRE_LOG[-3:]}) == 1   # one contextId per run
asked = run(orchestrator, "trends", show=False)
assert asked["status"] == "input-required" and len(asked["trace"]) == 1             # stopped after the first agent
failed = run(orchestrator, "please crash the research step", show=False)
assert failed["status"] == "failed" and "Stub Researcher" in failed["message"]
print("✓ 5. orchestrator: completes, stops for clarification, and reports a failing agent by name")
print("\nAll offline tests passed.")

✓ 1. discovery: Agent Cards served at /.well-known/agent-card.json and looked up by skill
✓ 2. message/send: returns a completed Task whose artifact has text and data parts
✓ 3. task states: input-required and failed are reported, not raised
✓ 4. errors: parse, unknown method, bad params, unknown task and unknown skill all handled
✓ 5. orchestrator: completes, stops for clarification, and reports a failing agent by name

All offline tests passed.


## Step 7: Connect to the LLM and web search

Set `PROVIDER` to `"openai"` or `"anthropic"`. All three agents use the same model, through three different frameworks.

In [23]:
import os

def _load(key):
    """Colab Secrets first, then an environment variable."""
    try:
        from google.colab import userdata
        value = userdata.get(key)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(key)


PROVIDER = "openai"   # or "openai"
MODEL = {"openai": "gpt-5-mini", "anthropic": "claude-sonnet-4-5"}[PROVIDER]
KEY_NAME = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[PROVIDER]

for key in (KEY_NAME, "TAVILY_API_KEY"):
    value = _load(key)
    if not value:
        raise RuntimeError(f"Add {key} to Colab Secrets (🔑) and turn on notebook access.")
    os.environ[key] = value       # LangChain, CrewAI and the SDKs all read keys from the environment

os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["OTEL_SDK_DISABLED"] = "true"
print(f"Using {PROVIDER} / {MODEL}")

Using openai / gpt-5-mini


## Step 8: The three real agents

Each agent is just a function `(text, data) -> AgentReply`. The A2A server from Step 3 makes it reachable. None of them knows the others exist.

**Agent 1: Task Extractor (LangChain).** Returns a validated `ResearchBrief` (task, 2–3 search queries, audience). If the request is too vague to research, it ends its task in the **`input-required`** state with a clarifying question.

In [24]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class ResearchBrief(BaseModel):
    task: str = Field(description="The user's request restated as one clear research task")
    search_queries: list[str] = Field(description="2 or 3 focused web search queries")
    audience: str = Field(description="Who the answer is for, e.g. 'enterprise leaders', or 'general' if not stated")
    needs_clarification: bool = Field(description="True only if the request is too vague to research at all")
    clarifying_question: str = Field(description="The question to ask the user if needs_clarification is true, else an empty string")


def make_chat_model():
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL)
    from langchain_anthropic import ChatAnthropic
    return ChatAnthropic(model=MODEL, max_tokens=2000)


extract_chain = ChatPromptTemplate.from_messages([
    ("system", "You turn a user's request into a research brief for a web researcher. "
               "Ask for clarification only if the request has no identifiable topic."),
    ("human", "{request}"),
]) | make_chat_model().with_structured_output(ResearchBrief)


def task_extractor(text: str, data: dict | None) -> AgentReply:
    brief = extract_chain.invoke({"request": text})
    if brief.needs_clarification:
        return AgentReply(state="input-required", text=brief.clarifying_question or "Could you say more about what you need?")
    brief.search_queries = [q for q in brief.search_queries if q.strip()][:3] or [brief.task]
    return AgentReply(text=brief.task, data=brief.model_dump())

**Agent 2: Researcher (CrewAI + Tavily).** A CrewAI agent searches the web with the brief's queries and returns structured findings, each with a `source_url`. Then **code checks the sources**: every URL the agent cites must be one that its search tool actually returned. Findings with invented URLs are dropped, and if none survive, the task **fails** rather than passing unsupported claims to the writer.

In [25]:
from crewai import LLM, Agent, Crew, Task
from crewai.tools import BaseTool
from tavily import TavilyClient


class Finding(BaseModel):
    point: str = Field(description="One specific, factual finding")
    source_url: str = Field(description="The URL of the search result this finding came from")


class ResearchFindings(BaseModel):
    findings: list[Finding]


class SearchInput(BaseModel):
    query: str = Field(description="A web search query")


class WebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for current information. Returns titles, URLs and snippets."
    args_schema: type[BaseModel] = SearchInput
    seen_urls: list = Field(default_factory=list)      # every URL this tool returned, for the source check

    def _run(self, query: str) -> str:
        results = TavilyClient(api_key=os.environ["TAVILY_API_KEY"]).search(query, max_results=5)["results"]
        self.seen_urls.extend(r["url"] for r in results)
        return "\n\n".join(f"{r['title']}\nURL: {r['url']}\n{r['content'][:600]}" for r in results)


def _norm(url: str) -> str:
    return url.strip().rstrip("/").lower()


def researcher(text: str, data: dict | None) -> AgentReply:
    brief = data or {"task": text, "search_queries": [text]}
    search = WebSearchTool()
    agent = Agent(role="Web Researcher",
                  goal="Find current, specific, well-sourced information",
                  backstory="You research with a search tool and never state anything you did not find in a result.",
                  tools=[search], llm=LLM(model=f"{PROVIDER}/{MODEL}"), allow_delegation=False, verbose=False)
    job = Task(description=(f"Research this task: {brief['task']}\n"
                            f"Start with these searches: {brief['search_queries']}\n"
                            "Return 5 findings. Each source_url must be copied exactly from a search result."),
               expected_output="5 findings, each with the URL it came from", agent=agent,
               output_pydantic=ResearchFindings)
    result = Crew(agents=[agent], tasks=[job], verbose=False, tracing=False).kickoff()
    findings = result.pydantic.findings if result.pydantic else ResearchFindings.model_validate_json(result.raw).findings

    # Source check, in code: keep only findings whose URL the search tool really returned
    seen = {_norm(u) for u in search.seen_urls}
    kept = [f.model_dump() for f in findings if _norm(f.source_url) in seen]
    dropped = len(findings) - len(kept)
    if not kept:
        return AgentReply(state="failed", text=f"None of the {len(findings)} findings could be traced to a search result.")
    return AgentReply(text=f"{len(kept)} sourced findings ({dropped} dropped: URL not in search results)",
                      data={"findings": kept, "dropped": dropped, "searched_urls": len(seen)})

**Agent 3: Writer (plain SDK, no framework).** Calls the OpenAI or Anthropic SDK directly. The sources are numbered, and the writer must cite them as `[1]`, `[2]`... Code then checks every citation: a number outside the list, or no citations at all, sends the draft back once with the problem; if it still fails, the task **fails**.

In [26]:
import re

CITATION_RE = re.compile(r"\[(\d+)\]")


def call_llm(prompt: str) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        reply = OpenAI().chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}])
        return reply.choices[0].message.content or ""
    import anthropic
    reply = anthropic.Anthropic().messages.create(model=MODEL, max_tokens=2000,
                                                  messages=[{"role": "user", "content": prompt}])
    return "".join(block.text for block in reply.content if block.type == "text")


def citation_problems(answer: str, n_sources: int) -> list:
    cited = {int(n) for n in CITATION_RE.findall(answer)}
    problems = [f"[{n}] is not a source number (valid: 1–{n_sources})" for n in sorted(cited) if not 1 <= n <= n_sources]
    if not cited:
        problems.append("the answer cites no sources")
    return problems


def writer(text: str, data: dict | None) -> AgentReply:
    findings = data["findings"]
    sources = list(dict.fromkeys(f["source_url"] for f in findings))          # unique, in order
    numbered = "\n".join(f"- {f['point']} [{sources.index(f['source_url']) + 1}]" for f in findings)
    prompt = (f"Task: {data['task']}\nAudience: {data.get('audience', 'general')}\n\n"
              f"Findings, with source numbers:\n{numbered}\n\n"
              "Write a concise answer (one short paragraph plus up to 5 bullet points) using ONLY these findings. "
              "Cite source numbers in square brackets, e.g. [1]. Do not add facts that are not in the findings.")
    answer = call_llm(prompt)
    problems = citation_problems(answer, len(sources))
    if problems:                                                             # one retry with the exact problem
        answer = call_llm(prompt + f"\n\nYour previous draft had problems: {'; '.join(problems)}. Rewrite it.")
        problems = citation_problems(answer, len(sources))
    if problems:
        return AgentReply(state="failed", text="Citation check failed: " + "; ".join(problems))
    return AgentReply(text=answer, data={"answer": answer, "sources": sources})


print("Agents defined: task_extractor (LangChain), researcher (CrewAI), writer (plain SDK)")

Agents defined: task_extractor (LangChain), researcher (CrewAI), writer (plain SDK)


## Step 9: Start the agents as A2A servers and discover them

The stub servers are stopped, the real agents start on new ports, and a fresh directory reads their Agent Cards. The orchestrator is built by the **same** `build_orchestrator` function the tests used.

In [27]:
for s in SERVERS:
    s.stop()
SERVERS = [
    A2AServer(make_agent_card("Task Extractor", "LangChain agent that turns requests into research briefs",
                              [EXTRACT_SKILL]), task_extractor).start(),
    A2AServer(make_agent_card("Researcher", "CrewAI agent that researches the web with source-checked findings",
                              [RESEARCH_SKILL]), researcher).start(),
    A2AServer(make_agent_card("Writer", "Framework-free agent that writes cited answers",
                              [WRITE_SKILL]), writer).start(),
]

directory = AgentDirectory()
for s in SERVERS:
    directory.register(s.url)            # discovery: everything the orchestrator knows comes from the cards
print("Registered agents:")
directory.show()

orchestrator = build_orchestrator(directory)

print("\nThe Researcher's Agent Card:")
print(json.dumps(fetch_agent_card(SERVERS[1].url), indent=2))

Registered agents:
  Task Extractor   http://127.0.0.1:41915/    skills: extract_research_brief
  Researcher       http://127.0.0.1:42619/    skills: web_research
  Writer           http://127.0.0.1:35335/    skills: write_cited_answer

The Researcher's Agent Card:
{
  "protocolVersion": "0.3.0",
  "name": "Researcher",
  "description": "CrewAI agent that researches the web with source-checked findings",
  "url": "http://127.0.0.1:42619/",
  "version": "1.0.0",
  "preferredTransport": "JSONRPC",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain",
    "application/json"
  ],
  "defaultOutputModes": [
    "text/plain",
    "application/json"
  ],
  "skills": [
    {
      "id": "web_research",
      "name": "Web research",
      "description": "Searches the web and returns findings, each tied to the URL it came from.",
      "tags": [
        "research",
        "web-search"
      ],
      "examples": [
        "Recen

## Step 10: Run a research question

The Researcher's step usually takes the longest, because it runs several web searches.

In [32]:
WIRE_LOG.clear()
result = run(orchestrator, "Can you summarize the latest trends in GenAI for enterprise?")

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): A2A calls:
  Task Extractor   extract_research_brief   → completed          9.6 s
  Researcher       web_research             → completed         99.5 s
  Writer           write_cited_answer       → completed         10.3 s

Status: completed

Enterprises are moving rapidly from experimentation to production: Gartner predicts over 80% will have used GenAI APIs/models or deployed GenAI apps by 2026 (up from <5% in 2023) [1], coinciding with a surge in enterprise AI spend to $37B in 2025 and broad uptake of RAG, fine‑tuning, tool calling, context engineering and reinforcement learning (with fine‑tuning and RL still niche for frontier teams) [5]; RAG in particular is a high‑growth area (market $1.92B in 2025, forecast $10.2B by 2030, ~39.66% CAGR) and has evolved into a sophisticated, multimodal enterprise intelligence architecture with hybrid retrieval and advanced filtering to improve factual, source‑grounded answers [2]

## Step 11: Look at the A2A messages on the wire

The exact JSON-RPC request the orchestrator sent to the Researcher, and the Task it got back. Long values are shortened for display.

In [33]:
def shorten(obj, limit: int = 160):
    if isinstance(obj, str):
        return obj if len(obj) <= limit else obj[:limit] + "…"
    if isinstance(obj, list):
        return [shorten(x, limit) for x in obj[:3]] + (["…"] if len(obj) > 3 else [])
    if isinstance(obj, dict):
        return {k: shorten(v, limit) for k, v in obj.items()}
    return obj


exchange = next(e for e in WIRE_LOG if e["agent_url"] == directory.find("web_research")["url"])
print("REQUEST →")
print(json.dumps(shorten(exchange["request"]), indent=2))
print("\n← RESPONSE")
print(json.dumps(shorten(exchange["response"]), indent=2))

REQUEST →
{
  "jsonrpc": "2.0",
  "id": "0ca9423b-21b5-4234-927b-8dcf9729b401",
  "method": "message/send",
  "params": {
    "message": {
      "kind": "message",
      "messageId": "43691810-6b2c-47eb-8bac-8e72cb5da77c",
      "role": "user",
      "parts": [
        {
          "kind": "text",
          "text": "Compile a concise summary of the latest trends in generative AI (GenAI) for enterprises, covering adoption patterns, key technologies (LLMs, multimodal, RAG, fi\u2026"
        },
        {
          "kind": "data",
          "data": {
            "task": "Compile a concise summary of the latest trends in generative AI (GenAI) for enterprises, covering adoption patterns, key technologies (LLMs, multimodal, RAG, fi\u2026",
            "search_queries": [
              "enterprise generative AI trends 2025 2026 report Gartner McKinsey Forrester \"generative AI\" enterprise adoption",
              "enterprise GenAI trends RAG retrieval-augmented generation LLMops fine-tuning mu

## Step 12: A vague request: the `input-required` state

With no identifiable topic, the Task Extractor should end its task as `input-required`. The orchestrator stops after one call and passes the agent's question back, instead of researching a guess. (The decision is the LLM's, so a very capable model may still find a topic to research.)

In [31]:
vague = run(orchestrator, "Can you look into it for me?")

A2A calls:
  Task Extractor   extract_research_brief   → input-required     9.2 s

Status: input-required
Task Extractor: What do you mean by “it”? Please provide: 1) the specific topic or question to research; 2) the purpose or how you will use the results; 3) desired depth/length (quick summary, detailed report, pros/cons, step-by-step, etc.); 4) any preferred or excluded sources (news, academic, blogs, official docs); 5) deadline or timeframe; and 6) intended audience/format (bullet list, executive summary, citations included). You can respond briefly or paste any relevant materials.


## Notes

- **What A2A gave this design:** the orchestrator depends on skills and message formats, not on LangChain, CrewAI or any SDK. Replacing the Researcher with an agent written in another language, or hosted by another team, means registering a new URL; nothing else changes. The offline tests in Step 6 show this by running the same orchestrator against stub agents.
- **Checks belong in code:** the Researcher drops findings whose URL its search never returned, and the Writer rejects citations that point to no source. Each agent is responsible for the quality of its own output before it crosses the wire.
- **Not covered here:** A2A also defines `message/stream` (progress updates over server-sent events), push notifications for long-running tasks, and authentication schemes declared in the Agent Card. The official `a2a-sdk` implements these; production agents would also run as separate services with authentication, rather than threads in one notebook.